<img src="https://www.funcionpublica.gov.co/documents/d/guest/logo-universidad-nacional" alt="Logo UNAL" width="600"/>

### **Universidad Nacional de Colombia sede Manizales**
#### Facultad de ingeniería y arquitectura
#### Departamento de ingeniería eléctrica, electrónica y computación
#### *Procesamiento del Lenguaje Natural*

#### Profesor: Lucas Iturriago
#### Monitora: Isabella Valero Mora - lvalerom@unal.edu.co

# 1. Laboratorio Frontend Reactivo y Despliegue MLOps

Utilizaremos Streamlit como un cliente delgado (Thin Client) encargado exclusivamente de capturar reportes técnicos en PDF, serializarlos y transmitirlos de forma síncrona hacia nuestro orquestador en la nube (n8n) mediante protocolos HTTP.

## 1.1. Configuración del Entorno e Instalación de Dependencias
Antes de levantar el servidor web local, debemos preparar la máquina virtual de Google Colab con las librerías del ecosistema de red, visualización analítica y la herramienta de tunelización reversa corporativa (Ngrok) para exponer nuestro puerto local a internet de forma ultra-estable.

## 1.2. Ejecución de Comandos de Consola
Ejecuta el siguiente bloque para instalar los paquetes requeridos e integrar el agente de Ngrok en el sistema operativo Linux de Colab:

In [ ]:
# Instalación de dependencias del Frontend, Red y wrapper de Tunelización
!pip install streamlit plotly requests pyngrok --quiet

# Instalación del agente nativo de Ngrok mediante el gestor de paquetes de Debian
!wget -qO - https://ngrok-agent.s3.amazonaws.com/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null \
  && echo "deb https://ngrok-agent.s3.amazonaws.com/ buster main" | sudo tee /etc/apt/sources.list.d/ngrok.list \
  && sudo apt-get update \
  && sudo apt-get install ngrok -y --quiet

## 1.3. Autenticación del Canal Cifrado (Ngrok Token)
Cada estudiante debe ingresar su Authtoken personal de Ngrok (obtenido de manera gratuita en ngrok.com) para firmar digitalmente el túnel TCP/TLS.

In [ ]:
# Token de autenticación de Ngrok para estabilización de canales de datos
NGROK_TOKEN = "INGRESE_AQUÍ_SU_AUTHTOKEN_DE_NGROK"
!ngrok config add-authtoken {NGROK_TOKEN}

# 2. Desarrollo del Artefacto Frontend (app.py)
Para garantizar un ciclo de vida desacoplado de grado de producción, la lógica de la interfaz se empaqueta en un script independiente de Python. Este script gobernará la serialización binaria en Base64 y la persistencia temporal de las gráficas reactivas mediante el estado de la sesión (st.session_state).

## 2.1. Escritura Automatizada del Script
Ejecuta la siguiente celda mágica para escribir el archivo app.py en el disco local de Colab:

In [ ]:
%%writefile app.py
import streamlit as st
import requests
import base64
import json
import plotly.graph_objects as go
from datetime import datetime

# =====================================================================
# CONFIGURACIÓN DE LA INTERFAZ DE USUARIO (FRONTEND)
# =====================================================================
st.set_page_config(page_title="UNAL - Monitoreo Inteligente", page_icon="📊", layout="wide")
st.title("📊 Monitor Inteligente de Reportes Técnicos")
st.markdown("Procesamiento del Lenguaje Natural — Universidad Nacional de Colombia\n---")

# Inicialización del buffer analítico local
if "history" not in st.session_state:
    st.session_state.history = []

# Barra Lateral - Conectividad MLOps
st.sidebar.header("Configuración de Red")
WEBHOOK_URL = st.sidebar.text_input("URL del Webhook de n8n (Test):", value="", placeholder="https://...")
SESSION_ID = st.sidebar.text_input("Identificador de Sesión (sessionId):", value="unal-clase-demo")

# UI Principal - Ingesta
st.subheader("1. Ingesta Multimodal de Reportes")
uploaded_file = st.file_uploader("Cargue el reporte técnico en formato PDF:", type=["pdf"])
chat_input = st.text_input("Instrucción para el Agente:", value="Procesa este reporte técnico y ejecuta la herramienta MCP.")

# =====================================================================
# PIPELINE DE TRANSMISIÓN Y PROCESAMIENTO (HTTP POST)
# =====================================================================
if uploaded_file is not None:
    st.success(f"📁 Archivo en buffer: {uploaded_file.name}")
    if st.button("🚀 Disparar Orquestación"):
        if not WEBHOOK_URL:
            st.error("Error: Ingrese la URL del Webhook de n8n en la barra lateral.")
        else:
            with st.spinner("Serializando bytes a Base64 y transmitiendo payload..."):
                try:
                    # Serialización y empaquetado del payload
                    file_bytes = uploaded_file.read()
                    base64_encoded = base64.b64encode(file_bytes).decode('utf-8')

                    payload = {
                        "sessionId": SESSION_ID,
                        "chatInput": chat_input,
                        "filename": uploaded_file.name,
                        "data_base64": base64_encoded
                    }

                    # Petición síncrona HTTP POST
                    response = requests.post(WEBHOOK_URL, json=payload, headers={"Content-Type": "application/json"})

                    if response.status_code == 200:
                        response_data = response.json()
                        st.balloons()
                        st.success("¡Pipeline completado! Datos guardados en la DB por el agente.")

                        # PARSEO ROBUSTO: Validar si la respuesta de n8n viene estructurada en una lista (All Given Items)
                        if isinstance(response_data, list):
                            datos_limpios = response_data[0]
                        else:
                            datos_limpios = response_data

                        # TRUCO DE ESTABILIDAD: Se usa st.code (texto seguro) en lugar de st.json
                        # para anular fallos por buffers incompletos del navegador en proxies masivos
                        with st.expander("🔍 Traza de Auditoría JSON (Payload Crudo)"):
                            st.code(json.dumps(response_data, indent=2), language="json")

                        # Extracción segura de la propiedad "output" devuelta por el Agente
                        output_texto = datos_limpios.get("output", "")
                        st.markdown(f"### Respuesta del Agente:\n> {output_texto}")

                        # Actualización de la telemetría local (Simulación reactiva del pipeline)
                        import random
                        eficiencia = 0.875 if "0.875" in output_texto or "87.5%" in output_texto else random.uniform(0.5, 0.95)
                        st.session_state.history.append({
                            "Timestamp": datetime.now().strftime("%H:%M:%S"),
                            "Reporte": uploaded_file.name.split('.')[0].upper(),
                            "Eficiencia (%)": round(eficiencia * 100, 2)
                        })
                    else:
                        st.error(f"Fallo en backend n8n. Código de Estado: {response.status_code}")
                except Exception as e:
                    st.error(f"Fallo crítico en parsing/transmisión: {str(e)}")

# =====================================================================
# UI PRINCIPAL - DASHBOARD DE OBSERVABILIDAD ANALÍTICA
# =====================================================================
st.markdown("---")
st.subheader("2. Dashboard de Telemetría en Tiempo Real")
col1, col2 = st.columns([1, 2])

with col1:
    st.markdown("### Historial de Ingestas")
    if st.session_state.history:
        st.dataframe(st.session_state.history, use_container_width=True)
    else:
        st.info("Esperando transacciones...")

with col2:
    st.markdown("### Índice de Eficiencia Operativa")
    if st.session_state.history:
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=[x["Timestamp"] for x in st.session_state.history],
            y=[x["Eficiencia (%)"] for x in st.session_state.history],
            mode='lines+markers', line=dict(color='#28669E', width=3)
        ))
        fig.update_layout(yaxis=dict(range=[0, 110]), template="plotly_white", margin=dict(l=20, r=20, t=20, b=20))
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("Esperando eventos de datos para proyectar telemetría.")

## 2.2. Despliegue del Servidor y Exposición del Túnel Estable (Ngrok)
Para interactuar con la interfaz gráfica web desde los navegadores locales sin micro-cortes, levantaremos Streamlit y crearemos el puente TLS dinámico mediante la API nativa de pyngrok.

In [ ]:
import os
from pyngrok import ngrok
import time

# 1. Inicialización del demonio de Streamlit en segundo plano del SO (Puerto 8501)
os.system("streamlit run app.py --server.port 8501 &")
time.sleep(3)  # Pequeña ventana de tiempo para asegurar el arranque del servidor web

# 2. Reseteo y limpieza de túneles huérfanos previos
ngrok.kill()

# 3. Lanzamiento del túnel HTTP seguro
try:
    public_url = ngrok.connect(8501, proto="http")
    print("=====================================================================")
    print("🚀 FRONTEND DESPLEGADO CON ÉXITO — ENTORNO MLOPS ESTABLE (SECURE-TLS)")
    print("=====================================================================")
    print("HAGA CLIC EN EL SIGUIENTE ENLACE PARA ABRIR LA INTERFAZ DE USUARIO:")
    print(f"👉 {public_url.public_url} 👈")
    print("=====================================================================")
except Exception as e:
    print(f"❌ Error al levantar la infraestructura de túneles en Ngrok: {e}")